In [1]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    "dbname": "voice_db",
    "user": "soubhikghosh",
    "password": "99Ghosh@",  # Provide a password if set
    "host": "localhost",
    "port": 5432
}

def connect_db():
    return psycopg2.connect(**DB_CONFIG, cursor_factory=RealDictCursor)

In [2]:
import random

def generate_unique_phone_number():
    """Generate a unique 10-digit phone number that does not exist in the database."""
    while True:
        phone_number = ''.join(random.choices("0123456789", k=10))
        conn = connect_db()
        cur = conn.cursor()
        cur.execute("SELECT phone_number FROM person_phone_mapping WHERE phone_number = %s", (phone_number,))
        if not cur.fetchone():
            conn.close()
            return phone_number
        conn.close()

def get_or_generate_phone_number(person_name):
    """
    Retrieve the phone number for the given person name if it exists.
    Otherwise, generate a unique phone number and insert a new record into person_phone_mapping.
    """
    conn = connect_db()
    cur = conn.cursor()
    # Try to fetch an existing phone number for the given person name.
    cur.execute("SELECT phone_number FROM person_phone_mapping WHERE person_name = %s", (person_name,))
    row = cur.fetchone()
    if row:
        # Depending on the cursor_factory, row may be a dict or a tuple.
        phone_number = row[0] if isinstance(row, tuple) else row["phone_number"]
        conn.close()
        return phone_number
    else:
        phone_number = generate_unique_phone_number()
        cur.execute(
            "INSERT INTO person_phone_mapping (person_name, phone_number) VALUES (%s, %s)",
            (person_name, phone_number)
        )
        conn.commit()
        conn.close()
        return phone_number


In [3]:
from speechbrain.pretrained import SpeakerRecognition
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import numpy as np
import librosa
import torch

# Load models
speaker_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="tmp")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
stt_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# Utility functions
def process_audio(audio_path):
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    return signal

def generate_embedding(signal, model=speaker_model):
    # Convert the signal to a numpy array and then to a tensor in a more efficient way
    signal = np.array(signal)  # Ensure signal is a numpy array
    embedding = model.encode_batch(torch.tensor(signal)).squeeze().tolist()
    return embedding


def transcribe_audio(audio_path):
    signal, sr = librosa.load(audio_path, sr=16000, mono=True)
    input_values = processor(signal, sampling_rate=sr, return_tensors="pt", padding=True).input_values
    logits = stt_model(input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription


INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
/var/folders/8c/29h2sjb912v9ht2qsjgcc05h0000gn/T/ipykernel_65458/3419363316.py:1: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import SpeakerRecognition
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb'

In [4]:
def register_user_in_db(person_name, phone_number, embedding, transcription):
    """
    Registers the user by inserting both the voice embedding and transcription into the database.
    """
    conn = connect_db()
    cur = conn.cursor()
    
    # Insert person data (if not already exists)
    cur.execute(
        "INSERT INTO person_phone_mapping (person_name, phone_number) VALUES (%s, %s) ON CONFLICT DO NOTHING",
        (person_name, phone_number),
    )
    
    # Insert the voice embedding along with transcription into the voice_embeddings table
    cur.execute(
        """
        INSERT INTO voice_embeddings (embedding, transcription, phone_number)
        VALUES (%s, %s, %s)
        """,
        (embedding, transcription, phone_number),
    )
    
    conn.commit()
    conn.close()


In [ ]:
import os
from flask import request, jsonify

def register_folder():
    data = request.get_json()
    folder_path = "../Data/"
    if not folder_path:
        return jsonify({"error": "Folder path is required."}), 400

    if not os.path.exists(folder_path) or not os.path.isdir(folder_path):
        return jsonify({"error": "Provided folder path does not exist or is not a directory."}), 400

    results = {}

    # Iterate over each subdirectory (each representing a person)
    for person_name in os.listdir(folder_path):
        person_dir = os.path.join(folder_path, person_name)
        if not os.path.isdir(person_dir):
            continue  # Skip files in the root folder
        
        # Get or generate the unique phone number for this person.
        phone_number = get_or_generate_phone_number(person_name)
        results[person_name] = {"registered_files": [], "errors": []}

        # Iterate over all .wav files in the subdirectory.
        for file in os.listdir(person_dir):
            if file.lower().endswith(".wav"):
                wav_path = os.path.join(person_dir, file)
                try:
                    # Process the audio file.
                    signal = process_audio(wav_path)
                    embedding = generate_embedding(signal)
                    transcription = transcribe_audio(wav_path)

                    # Register the user entry (inserts embedding and transcription).
                    register_user_in_db(person_name, phone_number, embedding, transcription)
                    results[person_name]["registered_files"].append(file)
                except Exception as e:
                    results[person_name]["errors"].append({file: str(e)})

    return jsonify(results), 200